# ESPN API breakdown

A hands-on tour of the unofficial ESPN endpoints this project uses (or may use), with the
fields that matter for **scores**, **game status** and **win probability**.

- Reference: [pseudo-r/Public-ESPN-API](https://github.com/pseudo-r/Public-ESPN-API)
- Verified against live responses on **2026-10-03**.

**Caveats.** These endpoints are undocumented by ESPN and can change without notice.
No rate limit is published, but heavy traffic can get blocked, so cache responses and
don't poll faster than you need.

**Run it** (from the repo root; adds Jupyter temporarily without changing project deps):

```bash
uv run --with jupyter jupyter lab backend/documentation/espn_api.ipynb
```

This notebook uses the *sync* `httpx.Client` because it's easier to explore with.
The app itself uses `httpx.AsyncClient` (see `src/client/espn/espn_client.py`).

## Quick look: one raw scoreboard, and where things live in it

A self-contained starting point: fetch one league's scoreboard and search the raw JSON.
Run these cells on their own; nothing else in the notebook is needed.

In [1]:
import httpx


def fetch_scoreboard(sport: str, league: str, day: str | None = None) -> dict:
    """GET one league's scoreboard. `day` is an Eastern date as YYYYMMDD; leave it out for today."""
    url = f"https://site.api.espn.com/apis/site/v2/sports/{sport}/{league}/scoreboard"
    params = {"dates": day} if day else {}
    response = httpx.get(url, params=params, timeout=15)
    response.raise_for_status()
    return response.json()


def find_value(obj, target, path: str = "") -> list[str]:
    """Every path in the JSON whose value equals `target`, e.g. 'events.0.status.type.state'."""
    hits = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            hits += find_value(value, target, f"{path}.{key}" if path else key)
    elif isinstance(obj, list):
        for index, value in enumerate(obj):
            hits += find_value(value, target, f"{path}.{index}")
    elif obj == target:
        hits.append(path)
    return hits

In [2]:
board = fetch_scoreboard("hockey", "nhl", "20261003")

print("top-level keys:", list(board))
league_info = board["leagues"][0]
print("leagues[0]:", {k: league_info[k] for k in ("id", "uid", "name", "abbreviation", "slug")})
first = board["events"][0]
print("events[0]: ", {k: first[k] for k in ("id", "uid", "name")})

top-level keys: ['leagues', 'events', 'provider']
leagues[0]: {'id': '90', 'uid': 's:70~l:90', 'name': 'National Hockey League', 'abbreviation': 'NHL', 'slug': 'nhl'}
events[0]:  {'id': '401891776', 'uid': 's:70~l:90~e:401891776', 'name': 'Chicago Blackhawks at Buffalo Sabres'}


In [3]:
for target in ("hockey", "nhl"):
    hits = find_value(board, target)
    print(f'"{target}" appears {len(hits)} times. First 3:')
    for hit in hits[:3]:
        print("   ", hit)

"hockey" appears 26 times. First 3:
    events.0.competitions.0.headlines.0.video.0.tracking.sportName
    events.0.competitions.0.highlights.0.tracking.sportName
    events.1.competitions.0.highlights.0.tracking.sportName
"nhl" appears 14 times. First 3:
    leagues.0.slug
    events.0.competitions.0.highlights.0.ad.sport
    events.1.competitions.0.highlights.0.ad.sport


**What this shows**

- There is **no sport field** at the top level or on each event.
- `leagues[0].slug` is the **league** (`"nhl"`). The sport only appears as a number in `uid` (`"s:70~l:90"`).
- `"hockey"` only shows up deep inside **tracking metadata** attached to videos, highlights or betting links
  (e.g. `...highlights.0.tracking.sportName`). Those blocks only exist when ESPN attaches media or odds to a game,
  so they are not a reliable place to read the sport from.
- The sport is already in the URL you requested (`/sports/hockey/nhl/scoreboard`), so the app knows it
  before ESPN replies. Pass it into the parser alongside `league`.

Use `find_value(board, "<some value>")` to locate any other field the same way.

## 0. The hosts

| Host | Base URL | Good for |
|---|---|---|
| Site API v2 | `https://site.api.espn.com/apis/site/v2/sports/{sport}/{league}/...` | scoreboard, game summary (one big JSON per game) |
| Core API v2 | `https://sports.core.api.espn.com/v2/sports/{sport}/leagues/{league}/...` | small, paginated resources: probabilities, odds, situation, plays |
| CDN | `https://cdn.espn.com/core/{league}/game?xhr=1&gameId={id}` | same data as the summary, wrapped in `gamepackageJSON` |

`{sport}/{league}` pairs used here: `football/nfl`, `baseball/mlb`, `hockey/nhl`.

In [1]:
import json

import httpx

SITE = "https://site.api.espn.com/apis/site/v2/sports"
CORE = "https://sports.core.api.espn.com/v2/sports"

# Finished games, so the output is the same every time you run this.
GAMES = {
    "nfl": ("football", "401872952"),  # Kansas City Chiefs at Miami Dolphins, 2026-09-27
    "mlb": ("baseball", "401908007"),  # Atlanta Braves at Los Angeles Dodgers, 2026-10-03
    "nhl": ("hockey", "401891776"),  # Chicago Blackhawks at Buffalo Sabres, 2026-10-03
}

http = httpx.Client(timeout=15, follow_redirects=True)


def get(url: str, **params) -> dict:
    response = http.get(url, params=params)
    response.raise_for_status()
    return response.json()


def peek(obj, max_chars: int = 800) -> None:
    """Pretty-print JSON, cut off after max_chars so big payloads stay readable."""
    text = json.dumps(obj, indent=2)
    print(text[:max_chars] + ("\n... (truncated)" if len(text) > max_chars else ""))

## 1. Scoreboard: every game for one day

`GET {SITE}/{sport}/{league}/scoreboard?dates=YYYYMMDD&limit=500`

- `dates` is a **single US-Eastern day**. Ranges (`20261001-20261005`) return HTTP 400 for NFL/MLB.
- Leave `dates` off to get "today" as ESPN defines it.
- This is the endpoint `EspnClient.get_scoreboard` calls today.

```
events[]
 ├─ id                        → matches.espn_id
 ├─ name                      → matches.matchup
 ├─ date                      → matches.date        (UTC, ISO string)
 ├─ status.type.state         → matches.status      "pre" | "in" | "post"
 ├─ status.type.detail        e.g. "Final", "Bot 7th", "2nd - 05:12"
 └─ competitions[0]
     └─ competitors[]
         ├─ homeAway          "home" | "away"   (don't assume index 0 is home)
         ├─ team.displayName  → matches.home_team / away_team
         └─ score             ⚠ a STRING, e.g. "4"
```

In [5]:
board = get(f"{SITE}/hockey/nhl/scoreboard", dates="20261003")
print(f"{len(board['events'])} NHL games on 2026-10-03\n")
for event in board["events"][:6]:
    sides = {c["homeAway"]: c for c in event["competitions"][0]["competitors"]}
    away, home = sides["away"], sides["home"]
    print(
        f"{event['id']}  {event['status']['type']['state']:<4}  "
        f"{away['team']['abbreviation']:>3} {away['score']:>2} @ "
        f"{home['team']['abbreviation']:<3} {home['score']:>2}   ({event['status']['type']['detail']})"
    )

13 NHL games on 2026-10-03

401891776  post  CHI  3 @ BUF  4   (Final)
401891822  post  MTL  5 @ PIT  6   (Final/OT)
401891826  post  WSH  1 @ TB   3   (Final)
401892434  post  OTT  3 @ TOR  2   (Final)
401892435  post  CAR  3 @ PHI  2   (Final/OT)
401892436  post  UTA  4 @ CBJ  1   (Final)


In [6]:
# One competitor, raw. Note "score" is a string.
competitor = board["events"][0]["competitions"][0]["competitors"][0]
peek({k: competitor[k] for k in ("homeAway", "winner", "score")} | {"team": {k: competitor["team"][k] for k in ("id", "abbreviation", "displayName")}})

{
  "homeAway": "home",
  "winner": true,
  "score": "4",
  "team": {
    "id": "2",
    "abbreviation": "BUF",
    "displayName": "Buffalo Sabres"
  }
}


## 2. Summary: everything about one game

`GET {SITE}/{sport}/{league}/summary?event={espn_id}`

One large JSON per game: header (status + scores), box score, plays, odds and, for some
sports, a `winprobability` array. The keys differ by sport, and the payload is **hundreds of KB**,
which matters if you plan to fetch it every few seconds.

In [7]:
summaries = {}
for league, (sport, event_id) in GAMES.items():
    response = http.get(f"{SITE}/{sport}/{league}/summary", params={"event": event_id})
    summaries[league] = data = response.json()
    print(f"{league}: {len(response.content) / 1024:,.0f} KB, winprobability entries: {len(data.get('winprobability', []))}")
    print("   keys:", ", ".join(sorted(data)), "\n")

nfl: 507 KB, winprobability entries: 169
   keys: againstTheSpread, article, boxscore, broadcasts, drives, format, gameInfo, header, injuries, leaders, meta, news, odds, pickcenter, scoringPlays, standings, tuneInPrompt, videos, wallclockAvailable, winprobability 

mlb: 937 KB, winprobability entries: 69
   keys: againstTheSpread, article, atBats, boxscore, broadcasts, format, gameInfo, header, injuries, meta, news, notes, odds, pickcenter, plays, playsMap, rosters, seasonseries, standings, videos, wallclockAvailable, winprobability 

nhl: 470 KB, winprobability entries: 0
   keys: againstTheSpread, article, boxscore, broadcasts, format, gameInfo, header, injuries, leaders, meta, news, odds, onIce, pickcenter, plays, seasonseries, standings, videos, wallclockAvailable 



In [8]:
# Status and scores live in the header, same shape for every sport.
for league, data in summaries.items():
    comp = data["header"]["competitions"][0]
    scores = {c["homeAway"]: c.get("score") for c in comp["competitors"]}
    print(f"{league}: {comp['status']['type']['state']:<4} {comp['status']['type']['detail']:<10} home {scores['home']} - away {scores['away']}")

nfl: post Final      home 10 - away 24
mlb: post Final      home 5 - away 3
nhl: post Final      home 4 - away 3


## 3. Win probability (from the summary)

`summary["winprobability"]` is a list, one entry per play:

| Field | Meaning |
|---|---|
| `homeWinPercentage` | home team's win probability, **0–1** (not 0–100) |
| `tiePercentage` | usually 0 |
| `playId` | which play this reading follows |

- There is **no away field**: away = `1 - homeWinPercentage - tiePercentage`.
- There is **no timestamp**. To know *when* a reading happened, look up its `playId` in the
  plays list and read that play's `wallclock` (UTC). MLB keeps plays at the top level
  (`plays[]`); NFL nests them inside drives (`drives.previous[].plays[]`).
- The first reading is usually the **pre-game** number, so it has no matching play.
- **NHL has no win probability at all** (empty here, HTTP 400 from the core endpoint).

In [9]:
peek(summaries["nfl"]["winprobability"][:3])

[
  {
    "homeWinPercentage": 0.1457,
    "tiePercentage": 0.0,
    "playId": "4018729521"
  },
  {
    "homeWinPercentage": 0.1322,
    "tiePercentage": 0.0,
    "playId": "40187295240"
  },
  {
    "homeWinPercentage": 0.099,
    "tiePercentage": 0.0,
    "playId": "40187295263"
  }
]


In [10]:
def plays_by_id(summary: dict) -> dict[str, dict]:
    """Index every play by id. MLB keeps plays at the top level; NFL nests them in drives."""
    plays = list(summary.get("plays", []))
    drives = summary.get("drives", {})
    for drive in drives.get("previous", []) + ([drives["current"]] if "current" in drives else []):
        plays.extend(drive.get("plays", []))
    return {play["id"]: play for play in plays}


def win_probability_timeline(summary: dict) -> list[dict]:
    plays = plays_by_id(summary)
    rows = []
    for wp in summary.get("winprobability", []):
        play = plays.get(wp["playId"], {})
        rows.append(
            {
                "wallclock": play.get("wallclock"),
                "period": play.get("period", {}).get("number"),
                "home_score": play.get("homeScore"),
                "away_score": play.get("awayScore"),
                "home_win": round(wp["homeWinPercentage"], 4),
                "away_win": round(1 - wp["homeWinPercentage"] - wp["tiePercentage"], 4),
            }
        )
    return rows


for league in ("nfl", "mlb"):
    timeline = win_probability_timeline(summaries[league])
    matched = sum(row["wallclock"] is not None for row in timeline)
    print(f"\n{league}: {len(timeline)} readings, {matched} matched to a play with a wallclock")
    for row in timeline[:3] + timeline[-2:]:
        print("  ", row)


nfl: 169 readings, 168 matched to a play with a wallclock
   {'wallclock': None, 'period': None, 'home_score': None, 'away_score': None, 'home_win': 0.1457, 'away_win': 0.8543}
   {'wallclock': '2026-09-27T17:02:37Z', 'period': 1, 'home_score': 0, 'away_score': 0, 'home_win': 0.1322, 'away_win': 0.8678}
   {'wallclock': '2026-09-27T17:03:17Z', 'period': 1, 'home_score': 0, 'away_score': 0, 'home_win': 0.099, 'away_win': 0.901}
   {'wallclock': '2026-09-27T19:59:28Z', 'period': 4, 'home_score': 10, 'away_score': 24, 'home_win': 0.001, 'away_win': 0.999}
   {'wallclock': '2026-09-27T19:59:28Z', 'period': 4, 'home_score': 10, 'away_score': 24, 'home_win': 0.0, 'away_win': 1.0}

mlb: 69 readings, 69 matched to a play with a wallclock
   {'wallclock': '2026-10-03T20:09:33Z', 'period': 1, 'home_score': 0, 'away_score': 0, 'home_win': 0.627, 'away_win': 0.373}
   {'wallclock': '2026-10-03T20:11:51Z', 'period': 1, 'home_score': 0, 'away_score': 0, 'home_win': 0.642, 'away_win': 0.358}
   {'wa

## 4. Win probability (from the Core API)

`GET {CORE}/{sport}/leagues/{league}/events/{id}/competitions/{id}/probabilities?limit=N`

- Small, **paginated** responses: `count`, `pageIndex`, `pageCount`, and `items[]`.
- Each item is only a **`$ref` link**; you fetch the link to get the values.
- Some `$ref`s point at `sports.core.api.espn.pvt`; replace `.pvt` with `.com`.
- Values include `awayWinPercentage` directly, plus `secondsLeft` and spread/total probabilities.
- Here the event id and competition id are the same number.

In [11]:
sport, event_id = GAMES["nfl"]
page = get(f"{CORE}/{sport}/leagues/nfl/events/{event_id}/competitions/{event_id}/probabilities", limit=3)
print({k: page[k] for k in ("count", "pageIndex", "pageSize", "pageCount")})
print("item 0 is just a link:", page["items"][0]["$ref"])

first = get(page["items"][0]["$ref"].replace(".pvt", ".com"))
peek({k: v for k, v in first.items() if k not in ("competition", "homeTeam", "awayTeam", "play", "$ref")})

{'count': 169, 'pageIndex': 1, 'pageSize': 3, 'pageCount': 57}
item 0 is just a link: http://sports.core.api.espn.com/v2/sports/football/leagues/nfl/events/401872952/competitions/401872952/probabilities/4018729521?lang=en&region=us
{
  "tiePercentage": 0.0,
  "homeWinPercentage": 0.1457,
  "awayWinPercentage": 0.8543000000000001,
  "lastModified": "2026-09-27T22:44Z",
  "sequenceNumber": "100",
  "source": {
    "id": "2",
    "description": "feed",
    "state": "full"
  },
  "secondsLeft": 0,
  "spreadCoverProbHome": 0.5,
  "spreadPushProb": 0.0,
  "totalOverProb": 0.5,
  "totalPushProb": 0.0
}


## 5. Sportsbook odds (Core API)

`GET {CORE}/{sport}/leagues/{league}/events/{id}/competitions/{id}/odds`

Available for **all three leagues, including NHL**. Prices are American moneylines.

**Concept: American odds → implied probability.** For a negative line *m*: `-m / (-m + 100)`
(e.g. -217 → 0.685). For a positive line *m*: `100 / (m + 100)` (e.g. +177 → 0.361).
The two sides add up to more than 1; the excess is the bookmaker's margin.

In [12]:
for league, (sport, event_id) in GAMES.items():
    odds = get(f"{CORE}/{sport}/leagues/{league}/events/{event_id}/competitions/{event_id}/odds")
    for item in odds.get("items", []):
        print(
            f"{league}: {item['provider']['name']:<12} {item.get('details') or '':<10} "
            f"home ML {item.get('homeTeamOdds', {}).get('moneyLine')}  away ML {item.get('awayTeamOdds', {}).get('moneyLine')}"
        )

nfl: DraftKings   KC -10     home ML 440  away ML -600
mlb: DraftKings   LAD -212   home ML -212  away ML 173
nhl: DraftKings   BUF -230   home ML -230  away ML 190


## 6. Live situation (Core API)

`GET {CORE}/{sport}/leagues/{league}/events/{id}/competitions/{id}/situation`

The current in-game state, and it differs by sport. For finished games it shows the final state.

In [13]:
for league, (sport, event_id) in GAMES.items():
    situation = get(f"{CORE}/{sport}/leagues/{league}/events/{event_id}/competitions/{event_id}/situation")
    print(f"{league}: {sorted(k for k in situation if k != '$ref')}")

nfl: ['awayTimeouts', 'distance', 'down', 'homeTimeouts', 'isRedZone', 'lastPlay', 'yardLine']
mlb: ['balls', 'batter', 'lastPlay', 'onSecond', 'outs', 'pitcher', 'situationNotes', 'strikes']
nhl: ['emptyNet', 'lastPlay', 'powerPlay']


## 7. Coverage (verified 2026-10-03)

| Data | NFL | MLB | NHL |
|---|---|---|---|
| Scoreboard + scores | ✅ | ✅ | ✅ |
| Summary `winprobability` | ✅ | ✅ | ❌ empty |
| Core `probabilities` | ✅ | ✅ | ❌ HTTP 400 |
| Core `predictor` (pre-game) | ✅ | ✅ | ❌ HTTP 400 |
| Core `odds` | ✅ | ✅ | ✅ |
| Core `situation` | down, distance, red zone | balls, strikes, outs, runners | power play, empty net |

## 8. curl cheat sheet

Run these in a terminal. `jq` trims the JSON to the parts that matter.
Swap `football/nfl` for `baseball/mlb` or `hockey/nhl`, and use an `espn_id` from your `matches` table.

**Today's games, status and scores (compact):**
```bash
curl -s "https://site.api.espn.com/apis/site/v2/sports/hockey/nhl/scoreboard" | jq '.events[] | {id, name, state: .status.type.state, detail: .status.type.detail, score: ([.competitions[0].competitors[] | {(.homeAway): .score}] | add)}'
```

**A specific day:**
```bash
curl -s "https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/scoreboard?dates=20261003" | jq '.events[] | {id, name, state: .status.type.state}'
```

**One game's status and score (summary header):**
```bash
curl -s "https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/summary?event=401908007" | jq '.header.competitions[0] | {state: .status.type.state, detail: .status.type.detail, score: ([.competitors[] | {(.homeAway): .score}] | add)}'
```

**Win probability readings (first 5):**
```bash
curl -s "https://site.api.espn.com/apis/site/v2/sports/football/nfl/summary?event=401872952" | jq '.winprobability[:5]'
```

**Plays with timestamps (MLB, last 5):**
```bash
curl -s "https://site.api.espn.com/apis/site/v2/sports/baseball/mlb/summary?event=401908007" | jq '.plays[-5:][] | {id, wallclock, period: .period.displayValue, homeScore, awayScore, text}'
```

**Core probabilities: page info, then the newest reading:**
```bash
curl -s "https://sports.core.api.espn.com/v2/sports/football/leagues/nfl/events/401872952/competitions/401872952/probabilities?limit=1000" | jq '{count, pageCount}'
```
```bash
curl -s "https://sports.core.api.espn.com/v2/sports/football/leagues/nfl/events/401872952/competitions/401872952/probabilities?limit=1000" | jq -r '.items[-1]["$ref"]' | xargs curl -s | jq '{homeWinPercentage, awayWinPercentage, tiePercentage, secondsLeft}'
```

**Sportsbook moneylines:**
```bash
curl -s "https://sports.core.api.espn.com/v2/sports/hockey/leagues/nhl/events/401891776/competitions/401891776/odds" | jq '.items[] | {provider: .provider.name, details, home: .homeTeamOdds.moneyLine, away: .awayTeamOdds.moneyLine}'
```

**Live situation:**
```bash
curl -s "https://sports.core.api.espn.com/v2/sports/baseball/leagues/mlb/events/401908007/competitions/401908007/situation" | jq 'del(.["$ref"], .lastPlay)'
```

**Save a whole response to explore it:**
```bash
curl -s "https://site.api.espn.com/apis/site/v2/sports/football/nfl/summary?event=401872952" -o nfl_summary.json && jq 'keys' nfl_summary.json
```

## 9. Where this meets the code

| Endpoint | Used by the app? | Fields read today |
|---|---|---|
| Site scoreboard | ✅ `EspnClient.get_scoreboard` → `parse_scoreboard` | `id`, `name`, `date`, `status.type.state`, competitors' `homeAway`, `team.displayName`, `score` |
| Site summary | not yet | |
| Core probabilities / odds / situation | not yet | |

Not yet decided (see `overview.md`, open design questions): which of these the app should
read for scores and win probability, how often, and how the readings are stored.

## 10. Walkthrough: a score's path from ESPN to the database

This section runs **the project's real code** (`parse_scoreboard`, `MatchCreate`,
`upsert_matches`) on one finished NHL game and shows what each layer produces.
Nothing is written to the database: step 3 swaps in a stand-in session that only
records the SQL statement.

```
ESPN JSON ──parse_scoreboard──► MatchCreate ──upsert_matches──► INSERT … ON CONFLICT DO UPDATE ──► matches row
  step 1          step 2           step 3                          step 4
```

_Saved output reflects the code as of 2026-10-04. Re-run this section after changing the code._

### Step 1: the raw ESPN fields

Each competitor carries `homeAway` and `score`. The score is a string, and it's `"0"`
(not missing) before a game starts.

In [14]:
event_id = GAMES["nhl"][1]
raw = get(f"{SITE}/hockey/nhl/scoreboard", dates="20261003")
event = next(e for e in raw["events"] if e["id"] == event_id)
for competitor in event["competitions"][0]["competitors"]:
    print(f"{competitor['homeAway']:<4} {competitor['team']['displayName']:<20} score = {competitor['score']!r}")

home Buffalo Sabres       score = '4'
away Chicago Blackhawks   score = '3'


### Step 2: the parser builds a `MatchCreate`

`parse_scoreboard` is a pure function, so you can call it on the JSON from step 1 directly.

In [15]:
from src.client.espn.parser import parse_scoreboard

games = parse_scoreboard(raw, "nhl")
game = next(g for g in games if g.espn_id == event_id)
peek(game.model_dump(mode="json"))
print("\nhome_score is a", type(game.home_score).__name__)

IndentationError: expected an indented block after 'for' statement on line 38 (parser.py, line 38)

### Step 3: do `MatchCreate` and the `Match` table agree?

`upsert_matches` turns each `MatchCreate` into a dict with `model_dump()` and inserts it.
Every key in that dict must be a column on `Match`, or SQLAlchemy refuses to build the SQL.

In [ ]:
from src.database.models import Match
from src.schemas.match import MatchCreate

schema_fields = set(MatchCreate.model_fields)
table_columns = set(Match.__table__.columns.keys())
print("in MatchCreate but not a column on Match:", sorted(schema_fields - table_columns) or "none")
print("columns on Match not in MatchCreate:   ", sorted(table_columns - schema_fields) or "none")
print("\nMatch column types:")
for column in Match.__table__.columns:
    print(f"  {column.name:<10} {str(column.type):<14} nullable={column.nullable}")

Now run the real `upsert_matches` with a stand-in session. The repository takes the session
as a parameter, so it can't tell the difference: it hands over its statement, and we print
the SQL instead of sending it to Postgres.

In [ ]:
from sqlalchemy.dialects import postgresql

from src.database.repositories import matches


class CaptureSession:
    """Stands in for AsyncSession: records the statement instead of running it."""

    statement = None

    async def execute(self, statement):
        self.statement = statement


session = CaptureSession()
await matches.upsert_matches(session, [game])
try:
    sql = str(session.statement.compile(dialect=postgresql.dialect()))
    print(sql)
except Exception as exc:
    sql = None
    print(f"Could not build the SQL -> {type(exc).__name__}: {exc}")

### Step 4: which columns update when the game already exists?

The upsert inserts a new game once. Every poll after that hits `ON CONFLICT (espn_id)`
and only the columns listed in `set_` are overwritten. A score that isn't listed is
saved once (usually `"0"`) and never changes.

In [ ]:
if sql is None:
    print("Fix step 3 first: the statement can't be built yet.")
else:
    updated = sql.split("DO UPDATE SET", 1)[1]
    for column in ("status", "home_score", "away_score"):
        print(f"{column:<11} updated on every poll: {column + ' =' in updated}")

### Step 5: strings vs numbers

Storing scores as text works for display, but Python and Postgres compare text
character by character.

In [ ]:
print('"10" > "9"  ->', "10" > "9")
print(" 10 >  9    ->", 10 > 9)
print('sorted(["10", "9", "2"]) ->', sorted(["10", "9", "2"]))

from pydantic import BaseModel


class Demo(BaseModel):
    score: int


print('\npydantic converts the string for you when the field is an int: Demo(score="4").score ->', repr(Demo(score="4").score))

In [ ]:
http.close()